# Estimating the Single-Company Delta_CoVaR for the Vietnam Tail Risk Indicator (VTRI)

In [ ]:
from google.colab import files
from google.colab import drive
import pandas as pd
import os
import io
import numpy as np
import statsmodels.api as sm
import warnings
warnings.filterwarnings("ignore")

In [ ]:
def import_data(file_path):
  try:
    drive.mount('/content/drive', force_remount=True)
    # Check if file exists
    if os.path.exists(file_path):
      df = pd.read_parquet(file_path)
      print(f"Loaded dataframe from Drive ({file_path})")
    else:
      raise FileNotFoundError(f"File not found at {file_path}")

  except Exception as e:
    print(f"Drive not available or file missing: {e}")
    print("Please upload dataframe manually.")
    uploaded = files.upload()

    # Automatically read the uploaded file
    file_name = list(uploaded.keys())[0]  # pick the first uploaded file
    try:
      df = pd.read_parquet(io.BytesIO(uploaded[file_name]))
    except:
      print("Wrong file extension. Parquet file required.")
    print(f"Loaded {file_name} from manual upload.")
    return df

Import the dataframe with the international markets data, df_glob, available in the <a href='https://github.com/SebastianoDenegri/vietnam-systemic-risk-index/tree/main/data/vsri_construction'>data/vsri_construction</a> subdirectory.

In [ ]:
# Import the Dataframe with the international market indexes
file_path = "..."
df_glob = import_data(file_path)

Drive not available or file missing: Error: credential propagation was unsuccessful
Please upload dataframe manually.


Saving df_glob.parquet to df_glob.parquet
Loaded df_glob.parquet from manual upload.


Clean and transform the data.

In [ ]:
# Data cleaning and transformation
df_glob.ffill(inplace=True)
df_glob_ret = df_glob.copy()
df_glob_ret = df_glob_ret.dropna()
df_glob_ret = df_glob_ret.set_index('Date')
df_glob_ret = np.log(df_glob_ret / df_glob_ret.shift(1))
df_glob_ret = df_glob_ret.dropna()
df_glob_ret.columns = df_glob_ret.columns.str.replace('_Close', '', regex=False)
df_glob_ret.columns = df_glob_ret.columns.str.replace('_close', '', regex=False)
df_glob_ret.columns = df_glob_ret.columns.str.replace('^', '', regex=False)
df_glob_ret

,VNINDEX,AXJO,FTSE,GSPC,GSPTSE,HSI,KS11,N225,STOXX50E,TWII,CSI300
Date,,,,,,,,,,,
2012-01-05,-0.022907,-0.010828,-0.007828,0.002939,0.000891,0.004587,-0.001330,-0.008376,-0.014635,0.006738,-0.009778
2012-01-06,-0.012425,-0.008290,0.004506,-0.002540,-0.003996,-0.011781,-0.011115,-0.011655,-0.007412,-0.001453,0.006226
2012-01-09,0.007662,-0.000755,-0.006642,0.002259,0.000664,0.014558,-0.009075,0.000000,-0.005322,-0.003865,0.033472
2012-01-10,0.015673,0.011335,0.014927,0.008847,0.006049,0.007318,0.014529,0.003796,0.026338,0.012028,0.032719
2012-01-11,0.007947,0.008466,-0.004557,0.000310,-0.000799,0.007740,-0.004147,0.003037,-0.003397,0.001300,-0.004809
...,...,...,...,...,...,...,...,...,...,...,...
2026-03-25,0.026534,0.021043,0.014109,0.005404,0.013712,0.010803,0.015772,0.028253,0.012117,0.025038,0.013924
2026-03-26,-0.008211,-0.003735,-0.013407,-0.017559,-0.015407,-0.019108,-0.032743,-0.002719,-0.014873,-0.003040,-0.013296
2026-03-27,0.016983,-0.001103,-0.000481,-0.016863,0.002293,0.003833,-0.003962,-0.004311,-0.010862,-0.006773,0.005576


Import the dataframe with the Vietnamese equity data, df_vn, available in the <a href='https://github.com/SebastianoDenegri/vietnam-systemic-risk-index/tree/main/data/vsri_construction'>data/vsri_construction</a> subdirectory.

In [ ]:
# Import the Dataframe with the VN data
df_vn = import_data(file_path)

Drive not available or file missing: Error: credential propagation was unsuccessful
Please upload dataframe manually.


Saving df_vn.parquet to df_vn.parquet
Loaded df_vn.parquet from manual upload.


Clean and transform the data.

In [ ]:
# Data cleaning and transformation
df_vn_price = df_vn.set_index('Date')[df_vn.columns[df_vn.columns.str.endswith('_close')]]
df_vn_price = df_vn_price*1000 #Cafef Data in thousand VND
df_vn_price.ffill(inplace=True)
df_vn_ret = np.log(df_vn_price/df_vn_price.shift(1))
df_vn_ret = df_vn_ret.dropna()
df_vn_ret.columns = df_vn_ret.columns.str.replace('_close', '', regex=False)
df_vn_ret

,NT2,PVT,PVS,PVD,REE,MBB,CTG,ACB,SHB,STB,...,DPM,HPG,HSG,NKG,FPT,DIG,DXG,KBC,KDH,VIC
Date,,,,,,,,,,,,,,,,,,,,,
2012-01-04,0.000000,0.029270,-0.063645,-0.048083,0.028472,0.000000,0.045085,0.014926,-0.009569,0.031416,...,-0.017805,-0.016261,-0.022990,-0.041031,0.000000,-0.019803,-0.017242,0.010030,0.035474,0.005109
2012-01-05,-0.088728,0.028438,-0.023122,-0.003917,0.000000,-0.022473,-0.004651,0.018349,-0.019418,-0.012680,...,-0.015083,-0.033336,-0.011696,0.005222,-0.022529,-0.020203,-0.017544,-0.020162,0.044568,-0.046945
2012-01-06,0.088728,-0.028438,-0.023670,0.000000,-0.017700,0.000000,0.016185,-0.003643,0.000000,0.012680,...,-0.009160,-0.034486,-0.011834,0.010363,0.010076,-0.025841,-0.008889,0.000000,-0.010309,-0.005355
2012-01-09,0.037041,0.000000,-0.015083,-0.012638,0.035091,0.015038,0.006857,-0.007326,0.000000,0.048081,...,0.024244,0.000000,0.023530,0.000000,0.002503,-0.031918,-0.018019,-0.018500,-0.013038,0.047191
2012-01-10,0.000000,0.028438,0.006061,0.028988,0.023851,0.014815,0.031393,0.007326,0.028988,0.035392,...,0.023670,0.000000,0.022990,0.000000,0.014889,-0.010870,0.044452,0.028632,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-03-25,0.067189,0.023311,0.028848,0.036634,0.067081,0.015504,0.014793,0.029853,0.037388,0.022765,...,0.020619,0.015297,0.003478,0.018833,0.020998,0.021979,0.032319,0.044216,0.033435,0.037203
2026-03-26,0.001803,0.031749,-0.004751,-0.001440,0.011220,-0.007722,-0.007369,-0.008439,0.009890,-0.026060,...,0.056213,-0.017225,0.006920,-0.003738,-0.037041,-0.018282,-0.021429,-0.006944,-0.015595,0.010050
2026-03-27,0.005391,-0.004474,0.018868,0.029811,0.000000,0.013475,0.029157,0.008439,0.016517,-0.011618,...,-0.009693,0.022902,0.027213,0.033152,0.025284,0.064308,0.052736,0.067366,0.036648,0.019803


Estimate the single-company $\Delta$CoVaRs, on 250-day rolling windows, with the Vietnamese financial system proxied by the VN-Index.

In [ ]:
# Align system and firm returns
y = df_glob_ret["VNINDEX"]
X_all = df_vn_ret
y, X_all = y.align(X_all, join="inner", axis=0)
#Calculate rolling single-institution DeltaCoVaR
window = 250
q = 0.05
# System returns (dependent variable)
# Store results
delta_covar = pd.DataFrame(index=df_vn_ret.index, columns=df_vn_ret.columns)
# Loop over each company
for col in df_vn_ret.columns:
  x_full = X_all[col]
  # Loop over time
  for i in range(window-1, len(df_vn_ret)):
    # Rolling window data
    y_win = y.iloc[i-window+1:i+1]
    x_win = x_full.iloc[i-window+1:i+1]
    # Add constant
    X = sm.add_constant(x_win)
    # Quantile regression (5%)
    model = sm.QuantReg(y_win, X).fit(q=q)
    # Firm VaR (5%) and median
    var_5 = x_win.quantile(q)
    median = x_win.quantile(0.5)
    # CoVaR at VaR and median
    covar_5 = model.params["const"] + model.params[col] * var_5
    covar_med = model.params["const"] + model.params[col] * median
    # deltaCoVaR
    delta_covar.iloc[i, df_vn_ret.columns.get_loc(col)] = covar_5 - covar_med

delta_covar.index = df_vn_ret.index

Export the results.

In [ ]:
delta_covar.to_csv("delta_covar.csv", index=True)
files.download("delta_covar.csv")
delta_covar.to_parquet("delta_covar.parquet", index=True)
files.download("delta_covar.parquet")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>